# METHOD 3 — SVM & Generalization
### A good classifier must work on drawings it has never seen.

This notebook is fully implemented. We will use a linear Support Vector Machine on HOG features, evaluate errors, and compare methods on identical validation examples.

**Presenter:** 7 minutes. Do not choose hyperparameters using the sealed test set.

In [ ]:
# Shared setup: works from local Jupyter or a fresh Google Colab runtime.
import subprocess, sys
from pathlib import Path
here = Path.cwd().resolve()
candidates = [here, here.parent, here / "doodle-ml-workshop", here.parent / "doodle-ml-workshop"]
ROOT = next((p for p in candidates if (p / "src" / "data.py").exists()), None)
if ROOT is None:
    checkout = Path("/content/Computer-Vision-Workshop")
    if not checkout.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "doodle-ai-workshop",
            "https://github.com/jaekop/Computer-Vision-Workshop.git", str(checkout)
        ], check=True)
    ROOT = checkout / "doodle-ml-workshop"
sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
print("Workshop ready at", ROOT)

In [ ]:
from src.data import load_data
from src.evaluation import show_examples, validation_accuracy, show_mistakes
from src.models import fit_model
import matplotlib.pyplot as plt
import numpy as np

data = load_data(cache_dir=ROOT / "data")
print("Categories:", ", ".join(data.class_names))
print("Training:", data.X_train.shape, "Validation:", data.X_val.shape)
print("Test is held back until the Final Challenge.")
show_examples(data.X_train, data.y_train, data.class_names, count=12)
plt.show()

## 1. SVM — find a separating boundary with a margin
An SVM prefers a wide margin between classes. Here we use a **linear** SVM in a HOG feature space, not a magical nonlinear image understanding system.

In [ ]:
svm_model = fit_model(data.X_train, data.y_train, "svm",
                      representation="hog", C=1.0)
print("HOG + linear SVM validation accuracy:",
      round(validation_accuracy(svm_model, data.X_val, data.y_val), 3))

## 2. Confusion matrix — what gets mixed up?
Overall accuracy hides which classes the model confuses. Read the rows as true classes and columns as predicted classes.

In [ ]:
from src.evaluation import show_confusion
from sklearn.metrics import classification_report
show_confusion(svm_model, data.X_val, data.y_val)
plt.show()
print(classification_report(
    data.y_val, svm_model.predict(data.X_val),
    target_names=list(data.class_names), zero_division=0
))

## 3. Inspect the actual failures
Do you see ambiguous objects, shifted strokes, incomplete shapes, or a class that is hard to distinguish?

In [ ]:
show_mistakes(svm_model, data.X_val, data.y_val, count=12)
plt.show()

## 4. Compare the four model families
All models see the same train/validation split. **Features are listed explicitly**, so this is a comparison of whole pipelines, not an isolated algorithm contest.

In [ ]:
comparisons = [
    ("KNN", "knn", "pixels", False),
    ("Decision Tree", "tree", "pixels", False),
    ("Random Forest", "forest", "pixels", False),
    ("Linear SVM + HOG", "svm", "hog", False),
]
results = []
for label, name, representation, center in comparisons:
    m = fit_model(data.X_train, data.y_train, name, representation, center)
    results.append((label, representation, validation_accuracy(m, data.X_val, data.y_val)))
for label, rep, score in results:
    print(f"{label:19s} | {rep:6s} | validation accuracy: {score:.3f}")
plt.barh([x[0] for x in results], [x[2] for x in results], color="#9B7BFF")
plt.xlim(0, 1); plt.xlabel("Validation accuracy")
plt.title("Measured on the same held-out validation split")
plt.show()

## 5. Cross-validation — one split can be noisy
Cross-validation repeats the train/validation experiment using different folds of the **training data only**. It helps estimate variation; it does not replace the final sealed test.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.features import build_features
from src.models import make_estimator
subset = np.arange(min(900, len(data.y_train)))
X_hog = build_features(data.X_train[subset], "hog")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_scores = cross_val_score(make_estimator("svm"), X_hog,
                              data.y_train[subset], cv=cv, scoring="accuracy")
print("3-fold training-only CV:", np.round(fold_scores, 3))
print("Mean:", round(float(fold_scores.mean()), 3))

## 6. A glimpse beyond the known classes
Our classifier **must choose one of six categories**. If you draw a dinosaur, it may confidently guess something else. Active learning asks which uncertain examples humans should label next. These are extensions, not claims that the current system solves unknown-class recognition.

### Copy this into the Final Challenge
- Use `fit_model(..., "svm", representation="hog", C=1.0)`.
- Generate a confusion matrix and inspect errors.
- Select a pipeline on validation. **Test only once at the end.**